In [13]:
import os 
from dotenv import load_dotenv

In [14]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages


from langgraph.prebuilt import ToolNode , tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random


In [15]:
llm = ChatOpenAI(
    model="poolside/laguna-xs-2.1:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [16]:
#toool 
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool 
def calculator(first_num: float , second_num: float, operation: str) -> dict:
    """Perform a basic arithmetic operation on two number.
    supported operations : add , sum , mul , div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {'error':"Division by zero is not allowed"}
            result = first_num/second_num
        else:
            return {"error":f"Unsupported operation '{operation}'"}
        
        return {"first_num":first_num , "second_num":second_num, "operation":operation, "result":result}
    
    except Exception as e:
        return {"error":str(e)}
    
    
@tool
def get_stock_price(symbol : str) -> dict:
    """Fetch latest stock price for a given symbol (e.g. 'AAPL','TSLA')
    using alpha vantage with api key in the url ."""
    
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=RSJK6ORTGNLO6MLI"
    r = requests.get(url)
    return r.json()


                

In [17]:
#make tool list 
tools = [get_stock_price , search_tool , calculator]

#make LLM tool aware
llm_with_tools = llm.bind_tools(tools)

In [18]:
#state 
class Chatstate(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [19]:
#graph Node
def chat_node(state:Chatstate):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}

tool_node = ToolNode(tools) #execute tool calls

In [20]:
#graph structure
graph = StateGraph(Chatstate)

graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)


In [21]:
graph.add_edge(START,"chat_node")

#If the LLM asked for the tool , go to the ToolNode; else finish 
graph.add_conditional_edges("chat_node",tools_condition)

graph.add_edge("tools", "chat_node")

In [22]:
chatbot = graph.compile()


In [23]:
#regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})
print(out['messages'][-1].content)

Hello! 👋 How can I assist you today? Whether it's a question, a task, or just a chat, I'm here for you!


In [106]:
#chat requiring tool 
out = chatbot.invoke({'messages':[HumanMessage(content="what is 2*3")]})
print(out['messages'][-1].content)

content='I can answer that directly without needing to use any tools.\n\n2 multiplied by 3 is 6.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 350, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 16, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'poolside/laguna-xs-2.1:free', 'system_fingerprint': None, 'id': 'gen-1783935460-GlTo75AOJ92doalF63nc', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f5ad6-e53f-7a43-9acc-3da6127dd636-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 350, 'output_tokens': 

In [12]:
#chat requiring tool 
out = chatbot.invoke({'messages':[HumanMessage(content="what is the stock price of apple . How much would it cost to purchase 50 shares?")]})
print(out['messages'][-1].content)

**Apple (AAPL) Stock Price:** $315.32 per share (as of July 10, 2026)

**Cost to purchase 50 shares:** $15,766.00

The calculation is: 50 shares × $315.32 per share = $15,766.00
